# AfriVoices — v4 PROPRE (corrige les 3 défauts qui ont tué le v4 précédent)

Le premier v4 a **dégradé** le modèle (macro 0.80 vs 0.52 pour v3) à cause de trois
erreurs cumulées :
1. **SpecAugment trop agressif** (F=15, T=25, 2 masques) → entrée mutilée, le modèle
   désapprend.
2. **Troncature 15s à l'entraînement** → cibles impossibles (texte complet, audio coupé).
3. **LR trop élevé** (2e-5) → fine-tuning instable depuis un bon modèle.

**Corrections ici :**
- SpecAugment **doux** (F=8, T=10, 1 masque chacun), **jamais à l'éval**.
- **Aucune troncature** : on **filtre** les audios > 20s en amont (cibles saines).
- **LR 1e-5** (la valeur qui marchait en v3).
- Éval avec collator propre (ni SpecAugment ni troncature).

**Test clé** : à la 1ère éval (pas 800), le WER doit être **≤ celui de v3**. S'il explose,
on arrête tout de suite — pas d'attente de 14h à l'aveugle.

Départ depuis **v3** (`baobab-asr-v3`), sortie **baobab-asr-v4** (écrase l'ancien v4 raté).

## 1. Dépendances

In [1]:
!pip install -q "transformers>=4.44,<5" "datasets>=3.5,<4" "accelerate>=0.33" \
    "jiwer>=3.0" torchaudio soundfile ffmpeg-python librosa huggingface_hub
print("ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
ok


## 2. Config + GPU

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
import numpy as np, torch
from huggingface_hub import login
# login("hf_...")   # décommente si besoin

V3_DIR="/content/drive/MyDrive/afrivoices/baobab-asr-v3"
V4_DIR="/content/drive/MyDrive/afrivoices/baobab-asr-v4"
WAV_DIR="/content/drive/MyDrive/afrivoices/wav_v4"
SAMPLING_RATE=16_000; MAX_AUDIO_S=20; SEED=42; TEMPERATURE=1.6
os.makedirs(V4_DIR, exist_ok=True); torch.manual_seed(SEED); np.random.seed(SEED)

# IDENTIQUES au run précédent (pour retrouver les mêmes WAV déjà sur le Drive)
N_PER_TYPE = {"som":12000, "kln":12000, "mas":10000, "luo":6000, "kik":4000}
SWA_PER_SHARD = 500

LANG_HOURS={"swa":2979,"som":1002,"kik":754,"luo":723,"kln":521,"mas":505}
LANGS=list(LANG_HOURS.keys())
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print(f"GPU {p.name} {p.total_memory/1e9:.0f}Go")
else: print("⚠️ pas de GPU")
def temperature_probs(h,T):
    a=np.array(list(h.values()),float); w=a**(1/T); w/=w.sum(); return dict(zip(h,w))
probs=temperature_probs(LANG_HOURS,TEMPERATURE)
print("probas:", {k:round(v,3) for k,v in probs.items()})

Mounted at /content/drive
GPU NVIDIA A100-SXM4-40GB 42Go
probas: {'swa': np.float64(0.332), 'som': np.float64(0.168), 'kik': np.float64(0.141), 'luo': np.float64(0.137), 'kln': np.float64(0.112), 'mas': np.float64(0.11)}


## 3. Processor v3 (vocab figé)

In [3]:
import re
from transformers import Wav2Vec2BertProcessor
processor=Wav2Vec2BertProcessor.from_pretrained(V3_DIR)
tokenizer=processor.tokenizer; feature_extractor=processor.feature_extractor
def clean_text(t):
    t=(t or "").lower().strip()
    t=re.sub(r"[^\w\sĩũáàâäéèêëíìîïóòôöúùûü']","",t,flags=re.UNICODE)
    return re.sub(r"\s+"," ",t)
print("vocab:", len(tokenizer))

vocab: 67


## 4. Reconstituer le dataset depuis les WAV déjà sur le Drive

Les ~60k WAV sont déjà dans `wav_v4` (générés au run précédent). On les réutilise :
on liste les fichiers, on récupère la langue depuis le nom, et le texte depuis...

⚠️ Le texte n'est PAS dans le nom de fichier. On doit le récupérer. Deux cas :
- si tu as encore `full`/`datasets_norm` en mémoire (même session) → saute à la §5.
- sinon → cette cellule relit les transcriptions depuis les parquets par appariement.

Le plus simple si la session est neuve : relancer les loaders §4 du notebook v4 initial
(ils réécrivent rien si les WAV existent déjà, ils reconstruisent juste les listes
audio_path/text/lang). Colle ici tes cellules load_ke + load_swahili + regroupement.
Puis reviens à la §5.

### Login Huggingface

In [ ]:
from huggingface_hub import login
login("hf_...")  # champ masqué — ne colle jamais ton token en clair

## 4.1 — 5 langues kényanes (gros volume)

In [5]:
import soundfile as sf, io, base64
from datasets import load_dataset, Dataset, Audio

KE_REPO="MCAA1-MSU/anv_data_ke"; KE_LANGS=["kik","luo","kln","mas","som"]
WAV_DIR="/content/drive/MyDrive/afrivoices/wav_v4"; os.makedirs(WAV_DIR, exist_ok=True)

def _decode(a):
    b=a.get("bytes")
    if b:
        try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
        except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
        return arr,sr
    return a["array"], a["sampling_rate"]

def load_ke(n_per_type, types=("unscripted","scripted")):
    paths,texts,langs=[],[],[]
    for code in KE_LANGS:
        tgt=n_per_type.get(code,0)
        if tgt<=0: continue
        for typ in types:
            patt=f"hf://datasets/{KE_REPO}/{code}/train/{typ}/audios/*.parquet"
            try: ds=load_dataset("parquet",data_files=patt,split="train",streaming=True)
            except Exception as e: print("skip",code,typ,str(e)[:50]); continue
            it=iter(ds); got=0; bad=0
            while got<tgt:
                try: ex=next(it)
                except StopIteration: break
                except Exception:
                    bad+=1
                    if bad>100: break
                    continue
                try:
                    txt=(ex.get("transcription") or "").strip()
                    if not txt: continue
                    fp=os.path.join(WAV_DIR,f"{code}_{typ}_{got}.wav")
                    if not os.path.exists(fp):
                        arr,sr=_decode(ex["audio"]); sf.write(fp,arr,sr)
                    paths.append(fp); texts.append(txt); langs.append(code); got+=1
                    if got%2000==0: print(f"  {code}/{typ}: {got}/{tgt}")
                except Exception: bad+=1; continue
            print(f"{code}/{typ}: +{got} (sautés {bad})")
    return Dataset.from_dict({"audio_path":paths,"text":texts,"lang":langs})

ke = load_ke(N_PER_TYPE)
print("total KE:", len(ke))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/194 [00:00<?, ?it/s]

  kik/unscripted: 2000/4000
  kik/unscripted: 4000/4000
kik/unscripted: +4000 (sautés 0)


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

  kik/scripted: 2000/4000
  kik/scripted: 4000/4000
kik/scripted: +4000 (sautés 0)


...

mas/scripted: +1839 (sautés 1)


Resolving data files:   0%|          | 0/134 [00:00<?, ?it/s]

  som/unscripted: 2000/12000
  som/unscripted: 4000/12000
  som/unscripted: 6000/12000
  som/unscripted: 8000/12000
  som/unscripted: 10000/12000
  som/unscripted: 12000/12000
som/unscripted: +12000 (sautés 0)


Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

som/scripted: +86 (sautés 1)
total KE: 60040


## 4.2 — Swahili (volume modéré, anti-oubli)

In [6]:
import json, tarfile, subprocess
from huggingface_hub import hf_hub_download
SWA_REPO="DigitalUmuganda/Afrivoice_Swahili"; SWA_TEXT=["normalized_transcription","transcription"]

def webm_to_wav(src,dst):
    subprocess.run(["ffmpeg","-y","-i",src,"-ar","16000","-ac","1",dst,"-loglevel","error"],check=True)

def load_swahili(domains=("agriculture","health","education","financial","government"),
                 split="train", shards_per_domain=1, max_per_shard=SWA_PER_SHARD):
    paths,texts,langs=[],[],[]
    for dom in domains:
        folder=f"{dom}_swahili_{split}"
        for n in range(shards_per_domain):
            try:
                man=hf_hub_download(SWA_REPO,f"{folder}/manifest_{n}.jsonl",repo_type="dataset")
                tarp=hf_hub_download(SWA_REPO,f"{folder}/audio/audio_{n}.tar.xz",repo_type="dataset")
            except Exception as e: print("skip",folder,str(e)[:50]); continue
            want={}
            with open(man,encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    d=json.loads(line); txt=next((d[k] for k in SWA_TEXT if d.get(k)),None)
                    key=d.get("key") or os.path.splitext(os.path.basename(d.get("audio_filepath","")))[0]
                    if txt and key: want[key]=txt.strip()
            ex=os.path.join(WAV_DIR,folder); os.makedirs(ex,exist_ok=True); done=0
            with tarfile.open(tarp,"r:xz") as t:
                for m in t:
                    if done>=max_per_shard: break
                    if not m.isfile() or not m.name.endswith(".webm"): continue
                    key=os.path.splitext(os.path.basename(m.name))[0]
                    if key not in want: continue
                    fo=t.extractfile(m)
                    if fo is None: continue
                    webm=os.path.join(ex,key+".webm"); wav=os.path.join(ex,key+".wav")
                    with open(webm,"wb") as o: o.write(fo.read())
                    try: webm_to_wav(webm,wav); os.remove(webm)
                    except Exception: continue
                    paths.append(wav); texts.append(want[key]); langs.append("swa"); done+=1
            print(f"{folder}: +{done}")
    return Dataset.from_dict({"audio_path":paths,"text":texts,"lang":langs})

swa = load_swahili()
print("total swa:", len(swa))

agriculture_swahili_train/manifest_0.jso(…):   0%|          | 0.00/6.60M [00:00<?, ?B/s]

agriculture_swahili_train/audio/audio_0.(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

agriculture_swahili_train: +500


health_swahili_train/manifest_0.jsonl:   0%|          | 0.00/7.09M [00:00<?, ?B/s]

health_swahili_train/audio/audio_0.tar.x(…):   0%|          | 0.00/1.35G [00:00<?, ?B/s]

...

financial_swahili_train/audio/audio_0.ta(…):   0%|          | 0.00/1.29G [00:00<?, ?B/s]

financial_swahili_train: +500


government_swahili_train/manifest_0.json(…):   0%|          | 0.00/7.11M [00:00<?, ?B/s]

government_swahili_train/audio/audio_0.t(…):   0%|          | 0.00/1.30G [00:00<?, ?B/s]

government_swahili_train: +500
total swa: 2500


## 4.3 — Regrouper + préparation légère (chemins + labels, PAS de features)

In [7]:
from datasets import concatenate_datasets
full = concatenate_datasets([ke, swa])
full = full.filter(lambda ex: ex["text"] and len(ex["text"].strip())>0)

def prep_light(ex):
    ex["labels"]=tokenizer(clean_text(ex["text"])).input_ids
    return ex
full = full.map(prep_light, num_proc=4, writer_batch_size=1000)
full = full.remove_columns([c for c in full.column_names if c not in ("audio_path","labels","lang")])

datasets_norm={}
for l in LANGS:
    datasets_norm[l]=full.filter(lambda x,c=l: x["lang"]==c)
    print(l, len(datasets_norm[l]))

Filter:   0%|          | 0/62540 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/62540 [00:00<?, ? examples/s]

Filter:   0%|          | 0/62540 [00:00<?, ? examples/s]

swa 2500


Filter:   0%|          | 0/62540 [00:00<?, ? examples/s]

...

luo 12000


Filter:   0%|          | 0/62540 [00:00<?, ? examples/s]

kln 16115


Filter:   0%|          | 0/62540 [00:00<?, ? examples/s]

mas 11839


## 4bis. FILTRER les audios > 20s (au lieu de tronquer)

C'est LA correction clé. On retire les audios trop longs du dataset — leurs cibles
seraient impossibles si on les tronquait. On calcule la durée via soundfile (rapide,
lit juste l'en-tête, pas tout le fichier).

In [8]:
import os
print("wav_v4 existe :", os.path.exists("/content/drive/MyDrive/afrivoices/wav_v4"))
print("nb WAV :", len(os.listdir("/content/drive/MyDrive/afrivoices/wav_v4")) if os.path.exists("/content/drive/MyDrive/afrivoices/wav_v4") else 0)
print("prepared_v3 existe :", os.path.exists("/content/drive/MyDrive/afrivoices/prepared_v3"))

wav_v4 existe : True
nb WAV : 60045
prepared_v3 existe : True


In [9]:
import soundfile as sf
def duree_s(path):
    try:
        info = sf.info(path)          # lit l'en-tête seulement
        return info.frames / info.samplerate
    except Exception:
        return 999.0                   # illisible -> exclu

for l in LANGS:
    if l not in datasets_norm: continue
    before = len(datasets_norm[l])
    datasets_norm[l] = datasets_norm[l].filter(
        lambda ex: duree_s(ex["audio_path"]) <= MAX_AUDIO_S, num_proc=4)
    print(f"{l}: {before} -> {len(datasets_norm[l])} (audios >{MAX_AUDIO_S}s retirés)")

Filter (num_proc=4):   0%|          | 0/2500 [00:00<?, ? examples/s]

swa: 2500 -> 1481 (audios >20s retirés)


Filter (num_proc=4):   0%|          | 0/12086 [00:00<?, ? examples/s]

som: 12086 -> 2001 (audios >20s retirés)


Filter (num_proc=4):   0%|          | 0/8000 [00:00<?, ? examples/s]

...

luo: 12000 -> 6699 (audios >20s retirés)


Filter (num_proc=4):   0%|          | 0/16115 [00:00<?, ? examples/s]

kln: 16115 -> 7037 (audios >20s retirés)


Filter (num_proc=4):   0%|          | 0/11839 [00:00<?, ? examples/s]

mas: 11839 -> 3381 (audios >20s retirés)


## 5. Split eval + mélange

In [10]:
from datasets import interleave_datasets
eval_per_lang={}; train_parts=[]
for l in LANGS:
    if l not in datasets_norm or not len(datasets_norm[l]): continue
    sp=datasets_norm[l].train_test_split(test_size=0.02, seed=SEED)
    train_parts.append(sp["train"]); eval_per_lang[l]=sp["test"]
order=[l for l in LANGS if l in eval_per_lang]
train_ds=interleave_datasets(train_parts, probabilities=[probs[l] for l in order],
                             seed=SEED, stopping_strategy="all_exhausted")
print("train:", len(train_ds), "| colonnes:", train_ds.column_names)
for l,d in eval_per_lang.items(): print("eval",l,len(d))

train: 60814 | colonnes: ['audio_path', 'lang', 'labels']
eval swa 30
eval som 41
eval kik 92
eval luo 134
eval kln 141
eval mas 68


## 6. Collators — SpecAugment DOUX (train) / PROPRE (éval)

SpecAugment doux : 1 masque fréquence (≤8) + 1 masque temps (≤10). **Aucune troncature**
(les audios longs ont été filtrés en §4bis). L'éval n'applique NI SpecAugment NI troncature.

In [11]:
from dataclasses import dataclass
import torch, soundfile as sf, librosa, numpy as np, random

def spec_augment_doux(feats, F=8, T=10):
    feats=feats.copy(); Tlen,D=feats.shape
    f=random.randint(0,F); f0=random.randint(0,max(0,D-f)); feats[:,f0:f0+f]=0   # 1 masque freq
    t=random.randint(0,T); t0=random.randint(0,max(0,Tlen-t)); feats[t0:t0+t,:]=0 # 1 masque temps
    return feats

@dataclass
class Collator:
    processor: object
    training: bool=True
    def __call__(self, features):
        arrays=[]
        for f in features:
            arr,sr=sf.read(f["audio_path"],dtype="float32")
            if arr.ndim>1: arr=arr.mean(axis=1)
            if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
            arrays.append(arr)                       # PAS de troncature
        b=self.processor.feature_extractor(arrays,sampling_rate=16000,return_tensors="pt",padding=True)
        if self.training:
            x=b["input_features"].numpy()
            for i in range(len(x)): x[i]=spec_augment_doux(x[i])
            b["input_features"]=torch.tensor(x)
        lb=self.processor.tokenizer.pad([{"input_ids":f["labels"]} for f in features],
                                        padding=True, return_tensors="pt")
        b["labels"]=lb["input_ids"].masked_fill(lb.attention_mask.ne(1),-100)
        return b

collator_train=Collator(processor, training=True)
collator_eval =Collator(processor, training=False)
print("collators: SpecAugment doux (train) / propre (eval), sans troncature")

collators: SpecAugment doux (train) / propre (eval), sans troncature


## 7. Fine-tuning PROPRE depuis v3

LR 1e-5, éval avec le collator propre via un Trainer sous-classé (indispensable pour une
métrique fiable). Sauvegarde + reprise auto. **Arrête si le WER à 800 pas > WER v3.**

In [13]:
import shutil, os, glob
V4_DIR = "/content/drive/MyDrive/afrivoices/baobab-asr-v4"

# lister ce qu'il y a
ckpts = glob.glob(f"{V4_DIR}/checkpoint-*")
print("checkpoints présents (du run raté):", ckpts)

# les supprimer pour repartir propre
for c in ckpts:
    shutil.rmtree(c)
    print("supprimé:", c)
# supprimer aussi les fichiers d'état du trainer à la racine s'il y en a
for f in glob.glob(f"{V4_DIR}/*.json") + glob.glob(f"{V4_DIR}/*.bin") + glob.glob(f"{V4_DIR}/*.safetensors"):
    os.remove(f); print("supprimé:", f)
print("\nV4_DIR nettoyé — le prochain run repartira de v3")
print("reste:", os.listdir(V4_DIR))

checkpoints présents (du run raté): ['/content/drive/MyDrive/afrivoices/baobab-asr-v4/checkpoint-1600', '/content/drive/MyDrive/afrivoices/baobab-asr-v4/checkpoint-15200']
supprimé: /content/drive/MyDrive/afrivoices/baobab-asr-v4/checkpoint-1600
supprimé: /content/drive/MyDrive/afrivoices/baobab-asr-v4/checkpoint-15200

V4_DIR nettoyé — le prochain run repartira de v3
reste: []


In [14]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer
from datasets import concatenate_datasets
import jiwer

model=Wav2Vec2BertForCTC.from_pretrained(V3_DIR, ctc_zero_infinity=True)
model.config.use_cache=False
print(f"Params {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

def preprocess_logits_for_metrics(logits, labels): return logits.argmax(dim=-1)
def compute_metrics(pred):
    ids=pred.predictions; labels=pred.label_ids
    labels[labels==-100]=processor.tokenizer.pad_token_id
    return {"wer": jiwer.wer(processor.batch_decode(labels, group_tokens=False),
                             processor.batch_decode(ids))}

# Trainer qui utilise le collator PROPRE pour l'éval (clé pour une métrique fiable)
class CleanEvalTrainer(Trainer):
    def get_eval_dataloader(self, eval_dataset=None):
        self.data_collator = collator_eval
        dl = super().get_eval_dataloader(eval_dataset)
        self.data_collator = collator_train
        return dl

eval_all=concatenate_datasets(list(eval_per_lang.values()))
use_bf16=torch.cuda.is_bf16_supported()

args=TrainingArguments(
    output_dir=V4_DIR, remove_unused_columns=False,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    per_device_eval_batch_size=2, eval_accumulation_steps=8,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant":False},
    learning_rate=1e-5, warmup_ratio=0.05, num_train_epochs=5,
    bf16=use_bf16, fp16=(not use_bf16),
    eval_strategy="steps", eval_steps=800,
    save_strategy="steps", save_steps=800, save_total_limit=3,
    logging_steps=100, load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False,
    dataloader_num_workers=8, report_to="none", seed=SEED)

trainer=CleanEvalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_all,
                data_collator=collator_train, compute_metrics=compute_metrics,
                preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                processing_class=processor)

from transformers.trainer_utils import get_last_checkpoint
last=get_last_checkpoint(V4_DIR) if os.path.isdir(V4_DIR) else None
if last: print("REPRISE depuis", last)
trainer.train(resume_from_checkpoint=last)
trainer.save_model(V4_DIR); processor.save_pretrained(V4_DIR)
print("v4 propre sauvegardé ->", V4_DIR)

Params 606M


Step,Training Loss,Validation Loss,Wer
800,0.742800,0.453589,0.500808
1600,0.742400,0.456346,0.508017
2400,0.726600,0.452380,0.493599
3200,0.731500,0.439927,0.496706
4000,0.681200,0.439920,0.488005
4800,0.710900,0.435119,0.488129
5600,0.672000,0.447413,0.495463
6400,0.647000,0.430652,0.486140
7200,0.685800,0.430513,0.493101
8000,0.629600,0.425842,0.483157


v4 propre sauvegardé -> /content/drive/MyDrive/afrivoices/baobab-asr-v4


## 8. Éval propre par langue (comparer à v3)

Mesure fiable : audios entiers, sans SpecAugment. À lancer sur le meilleur checkpoint.

In [16]:
import torch, jiwer, soundfile as sf, librosa, numpy as np
from transformers import Wav2Vec2BertForCTC

def eval_propre(model_dir, eval_per_lang, processor, max_par_langue=60):
    dev="cuda" if torch.cuda.is_available() else "cpu"
    m=Wav2Vec2BertForCTC.from_pretrained(model_dir).to(dev).eval()
    res={}
    for lang,d in eval_per_lang.items():
        preds,refs=[],[]; n=min(len(d),max_par_langue)
        with torch.no_grad():
            for j in range(n):
                ex=d[j]; arr,sr=sf.read(ex["audio_path"],dtype="float32")
                if arr.ndim>1: arr=arr.mean(axis=1)
                if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
                fe=processor.feature_extractor([arr],sampling_rate=16000,return_tensors="pt",padding=True)
                fe={k:v.to(dev) for k,v in fe.items()}
                preds.append(processor.batch_decode(torch.argmax(m(**fe).logits,dim=-1))[0])
                lab=[t for t in ex["labels"] if t!=-100]
                refs.append(processor.tokenizer.decode(lab, group_tokens=False))
        res[lang]=jiwer.wer(refs,preds); print(f"  {lang}: {res[lang]:.3f} (n={n})")
    macro=sum(res.values())/len(res); print(f"  Macro: {macro:.3f}"); return res,macro

print("=== v3 (référence) ==="); eval_propre(V3_DIR, eval_per_lang, processor)
from transformers.trainer_utils import get_last_checkpoint
print("=== v4 (best model) ===")
eval_propre(V4_DIR, eval_per_lang, processor)

=== v3 (référence) ===
  swa: 0.081 (n=30)
  som: 0.649 (n=41)
  kik: 0.365 (n=60)
  luo: 0.275 (n=60)
  kln: 0.733 (n=60)
  mas: 0.583 (n=60)
  Macro: 0.448
=== v4 (best model) ===
  swa: 0.097 (n=30)
  som: 0.638 (n=41)
  kik: 0.270 (n=60)
  luo: 0.280 (n=60)
  kln: 0.669 (n=60)
  mas: 0.534 (n=60)
  Macro: 0.415


({'swa': 0.09684947491248541,
  'som': 0.6376811594202898,
  'kik': 0.2702104097452935,
  'luo': 0.2796208530805687,
  'kln': 0.6685082872928176,
  'mas': 0.5341614906832298},
 0.4145052791891142)